# DNA26-1804 — ALTER `derivativeactive.dsettlementexposure` add case/phase + isPromiseActive

One-off deployment. Adds 7 new columns to `{env}-cdh.derivativeactive.dsettlementexposure`
in the expected order (appended after the current last column `MltAmt`, matching the
`DSettlementExposure.ipynb` schema string and both profits/qca selects).

New columns (in order):
`PromiseCaseID` string, `PromiseCasePurpose` string, `PromiseCaseDescription` string,
`PromiseCaseArrivalDate` date, `SettlementPhase` string, `SettlementJustification` string,
`isPromiseActive` int.

Run ONCE per environment AFTER the curated `settlementexposure` ALTER + reload (so the
source columns exist) and BEFORE `DSettlementExposure` runs with the new code.
Set the `env` widget (qa / p). Idempotent: `ADD COLUMNS IF NOT EXISTS`.

Note: `isPromiseActive` = IT TABLE_STATUS='1' perimeter; populated =1 for profits,
NULL for qca by the derivative notebook.

In [0]:
dbutils.widgets.text('env', 'qa', 'Environment prefix (qa / p)')
env = dbutils.widgets.get('env').strip()
assert env in ('qa', 'p'), f"Unexpected env: {env!r} (expected 'qa' or 'p')"
table = f"`{env}-cdh`.derivativeactive.dsettlementexposure"
print(f"Target table: {table}")

In [0]:
before = [f.name for f in spark.table(table).schema.fields]
print(f"{len(before)} columns before")
print([c for c in before if c in (
    'MltAmt', 'PromiseCaseID', 'PromiseCasePurpose', 'PromiseCaseDescription',
    'PromiseCaseArrivalDate', 'SettlementPhase', 'SettlementJustification', 'isPromiseActive')])

In [0]:
spark.sql(f"""
    ALTER TABLE {table} ADD COLUMNS IF NOT EXISTS (
        PromiseCaseID string,
        PromiseCasePurpose string,
        PromiseCaseDescription string,
        PromiseCaseArrivalDate date,
        SettlementPhase string,
        SettlementJustification string,
        isPromiseActive int
    )
""")
print("ALTER done")

In [0]:
after = [f.name for f in spark.table(table).schema.fields]
print(f"{len(after)} columns after")
expected_tail = [
    'PromiseCaseID', 'PromiseCasePurpose', 'PromiseCaseDescription',
    'PromiseCaseArrivalDate', 'SettlementPhase', 'SettlementJustification', 'isPromiseActive']
print('tail =', after[-7:])
assert after[-7:] == expected_tail, f"Unexpected tail order: {after[-7:]}"
print('OK — 7 columns added in expected order')